In [2]:
import pandas as pd
import torch
from sklearn.utils import shuffle

In [12]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "Sota"or cwd.name == "ML100K":
    chdir("/Users/haowen/Documents/Decentral RS/fed-learning-main")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code/Sota/ML100K
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [14]:
train = pd.read_csv("dataset/ml100k_train.csv")
test = pd.read_csv("dataset/ml100k_test.csv")
train.head()

,user_id,item_id,rating,rating.1
0,207,87,5,5
1,675,900,4,4
2,373,230,2,2
3,377,564,3,3
4,725,843,3,3


In [16]:
train.head()

,user_id,item_id,rating,rating.1
0,207,87,5,5
1,675,900,4,4
2,373,230,2,2
3,377,564,3,3
4,725,843,3,3


In [18]:
train.shape

(75000, 4)

In [20]:
test.shape

(24929, 4)

In [22]:
rating_matrix = train.pivot(index='user_id', columns='item_id', values='rating')
n_users, n_movies = rating_matrix.shape
# Scaling ratings to between 0 and 1, this helps our model by constraining predictions
min_rating, max_rating = train['rating'].min(), train['rating'].max()
rating_matrix = (rating_matrix - min_rating) / (max_rating - min_rating)
print(rating_matrix)
print(n_users*n_movies-rating_matrix.isnull().values.sum())

item_id  0     1     2     3     4     5     6     7     8     9     ...  \
user_id                                                              ...   
0        1.00   0.5  0.75  0.50   NaN   1.0   NaN   0.0   NaN  0.50  ...   
1         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  0.25  ...   
2         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
3         NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4        0.75   0.5   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
938       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   1.0   NaN  ...   
939       NaN   NaN   NaN  0.25   NaN   NaN  0.75   1.0   NaN   NaN  ...   
940       NaN   NaN   NaN   NaN   NaN   NaN  0.75   NaN   NaN   NaN  ...   
941       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
942       NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

item_id  16

In [24]:
# Replacing missing ratings with -1 so we can filter them out later

rating_matrix[rating_matrix.isnull()] = -1
rating_matrix = torch.FloatTensor(rating_matrix.values)
print(rating_matrix)

tensor([[ 1.0000,  0.5000,  0.7500,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        ...,
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
        [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]])


In [26]:
class PMF(torch.nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, matrix, u_features, v_features):
        non_zero_mask = (matrix != -1).type(torch.FloatTensor)
        predicted_rating = torch.sigmoid(torch.mm(u_features, v_features.t()))
        
        return predicted_rating

In [28]:
class PMFLoss(torch.nn.Module):
    def __init__(self, lam_u=0.3, lam_v=0.3):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
    
    def forward(self, matrix, predicted, u_features, v_features):
        non_zero_mask = (matrix != -1).type(torch.FloatTensor)
        diff = (matrix - predicted)**2
        prediction_error = torch.sum(diff*non_zero_mask)

        u_regularization = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_regularization = self.lam_v * torch.sum(v_features.norm(dim=1))
        
        # u_regularization = self.lam_u * torch.sum(u_features**2)
        # v_regularization = self.lam_v * torch.sum(v_features**2)   

        return prediction_error + u_regularization + v_regularization

In [30]:
# criterion = PMFLoss()
# loss = criterion(rating_matrix, user_features, movie_features)

In [32]:
# PMF Actual training loop now
pmf = PMF()
pmfloss = PMFLoss(lam_u=0.1, lam_v=0.1)
num_epoch = 100
latent_vectors = 20
user_features = torch.randn(n_users, latent_vectors, requires_grad=True)
user_features.data.mul_(0.01)
movie_features = torch.randn(n_movies, latent_vectors, requires_grad=True)
movie_features.data.mul_(0.01)

# pmferror = PMFLoss(lam_u=0.1, lam_v=0.1)
optimizer = torch.optim.Adam([user_features, movie_features], lr=0.01)
for step, epoch in enumerate(range(num_epoch)):
    optimizer.zero_grad()
    pred = pmf(rating_matrix, user_features, movie_features)
    loss = pmfloss(rating_matrix, pred, user_features, movie_features)
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"Step {step}, {loss:.3f}")

Step 0, 7268.239
Step 10, 6920.357
Step 20, 5284.296
Step 30, 4488.768
Step 40, 4135.469
Step 50, 3847.685
Step 60, 3606.429
Step 70, 3361.009
Step 80, 3105.207
Step 90, 2853.848


In [33]:
test_rating_matrix = test.pivot(index='user_id', columns='item_id', values='rating')

test_rating_matrix[test_rating_matrix.isnull()] = -1
test_rating_matrix = torch.FloatTensor(test_rating_matrix.values)
test_rating_matrix.shape

torch.Size([943, 1408])

In [36]:
train_items = train["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
test_items = test["item_id"].drop_duplicates().sort_values().reset_index(drop=True)
# Create a mapping from item_id → index in train_items
item_to_index = pd.Series(train_items.index.values, index=train_items.values)

# Map test_items to train indices
test_indices_in_train = test_items.map(item_to_index)

print(test_indices_in_train)

0          0
1          1
2          2
3          3
4          4
        ... 
1403    1593
1404    1596
1405    1606
1406    1608
1407    1620
Name: item_id, Length: 1408, dtype: int64


In [38]:
## test General PMF
non_zero_mask = (test_rating_matrix != -1).type(torch.FloatTensor)
num = torch.sum(non_zero_mask)
# print(num)
# print(non_zero_mask)
predicted_ratings = torch.sigmoid(torch.mm(user_features, movie_features.t()))
predicted_ratings = predicted_ratings[:,test_indices_in_train]
# predicted_ratings = non_zero_mask * predicted_ratings
predicted_ratings = (predicted_ratings*(max_rating - min_rating) + min_rating)*non_zero_mask
actual_ratings = test_rating_matrix*non_zero_mask
# print(predicted_ratings)
# print(actual_ratings)

AE_diff = torch.abs(predicted_ratings - actual_ratings)
SE_diff = (predicted_ratings - actual_ratings)**2
# print(AE_diff)
# print(SE_diff)
test_MAE = torch.sum(AE_diff)/num
test_RMSE = torch.sqrt(torch.sum(SE_diff)/num)
# print(torch.count_nonzero(predicted_ratings).item())
# print(torch.count_nonzero(actual_ratings).item())
print('test_MAE =', test_MAE.data.numpy())
print('test_RMSE =', test_RMSE.data.numpy())

RuntimeError: The size of tensor a (1628) must match the size of tensor b (1408) at non-singleton dimension 1